# Create graph figure

### Import libraries

In [ ]:
import igraph
import pandas as pd

### Import data

In [2]:
df = pd.read_csv('../raw/databases_rna_sugarcane.csv')

In [ ]:
def create_edges_merge(df: pd.DataFrame) -> list:
    """
    Create edges based on shared ontology terms using a self-join approach.
    Args:
        df (pd.DataFrame): The input DataFrame containing BioProject and ontology columns.
    Returns:
        list: A list of tuples representing edges between BioProjects that share ontology terms.
    """
    edges = set()
    ontology_cols = [col for col in df.columns if 'ontology' in col]
    
    for col in ontology_cols:
        # Self-join on each ontology column
        merged = df[['BioProject', col]].merge(
            df[['BioProject', col]], 
            on=col, 
            how='inner'
        )
        # Filter to keep only run1 < run2 to avoid duplicates
        merged = merged[merged['BioProject_x'] < merged['BioProject_y']]
        edges.update(zip(merged['BioProject_x'], merged['BioProject_y']))
    
    return list(edges)

In [ ]:
def create_edges_merge_by_ontology_type(df: pd.DataFrame) -> list:
    """
    Create edges based on shared ontology terms using a self-join approach.
    Args:
        df (pd.DataFrame): The input DataFrame containing BioProject and ontology columns.
    Returns:
        list: A list of tuples representing edges between BioProjects that share ontology terms.
    """
    edges = {}  # (BioProject_x, BioProject_y) -> list of ontology cols
    ontology_cols = [col for col in df.columns if 'ontology' in col]
    
    for col in ontology_cols:
        # Self-join on each ontology column
        merged = df[['BioProject', col]].merge(
            df[['BioProject', col]], 
            on=col, 
            how='inner'
        )
        # Filter to keep only run1 < run2 to avoid duplicates
        merged = merged[merged['BioProject_x'] < merged['BioProject_y']]
        
        for bp_x, bp_y in zip(merged['BioProject_x'], merged['BioProject_y']):
            key = (bp_x, bp_y)
            if key not in edges:
                edges[key] = set()
            edges[key].add(col)
    
    # Return list of (BioProject_x, BioProject_y, [ontologies])
    return [(a, b, sorted(cols)) for (a, b), cols in edges.items()]

In [ ]:
def create_edges_merge_by_ontology_value(df: pd.DataFrame) -> list:
    """
    Create edges based on shared ontology terms using a self-join approach, grouping by ontology values.
    Args:
        df (pd.DataFrame): The input DataFrame containing BioProject and ontology columns.
    Returns:
        list: A list of tuples representing edges between BioProjects that share ontology terms, grouped by ontology values.
    """
    edges = {}
    ontology_cols = [col for col in df.columns if 'ontology' in col]
    
    for col in ontology_cols:
        merged = df[['BioProject', col]].dropna(subset=[col]).merge(  # drop NaN before merging
            df[['BioProject', col]].dropna(subset=[col]), 
            on=col, 
            how='inner'
        )
        merged = merged[merged['BioProject_x'] < merged['BioProject_y']]
        
        for bp_x, bp_y, ontology_value in zip(merged['BioProject_x'], merged['BioProject_y'], merged[col]):
            key = (bp_x, bp_y)
            if key not in edges:
                edges[key] = set()
            edges[key].add(ontology_value)
    
    return [(a, b, sorted(cols)) for (a, b), cols in edges.items()]

In [ ]:
def create_edges_merge_ontology2node(df: pd.DataFrame) -> set:
    """
    Create edges between BioProjects and ontology terms, treating ontologies as nodes.
    Args:
        df (pd.DataFrame): The input DataFrame containing BioProject and ontology columns.
    Returns:
        list: A list of tuples representing edges between BioProjects and ontology terms.
    """
    edges = set()
    ontology_cols = [col for col in df.columns if 'ontology' in col]
    
    for col in ontology_cols:
        merged = df[['BioProject', col]].dropna(subset=[col]).drop_duplicates()
        
        for bp, ont in zip(merged['BioProject'], merged[col]):
            for o in ont.split(','):
                edges.add((bp, o.strip()))

    return edges

In [ ]:
edges = create_edges_merge_ontology2node(df) # or other functions created above
print(f"Number of edges: {len(edges)}")

Number of edges: 374


In [ ]:
graph = igraph.Graph.TupleList(
    edges,
    directed=False,
    vertex_name_attr='BioProject',
    edge_attrs=['ontologies'] # just for the functions that return ontologies as edge attributes
)

In [6]:
graph.summary()

'IGRAPH U--- 196 374 -- \n+ attr: BioProject (v), ontologies (e)'

In [ ]:
graph.es['ontologies'] = [o for o in graph.es['ontologies']] # just for the functions that return ontologies as edge attributes

# Export graphml to use in gephi or cytoscape
graph.write_graphml('../raw/graph_sugarcane.graphml')